# Data-Driven Incentive Design
### Optimizing incentives for taxi drivers using causal estimates of supply elasticity
#### Julian Streyczek, June 2026

This is a case study that exemplifies how to combine user data, causal inference, and optimization to design an incentive scheme. The example uses public data on taxi trips in San Francisco and designs a simple tiered rewards scheme to increases the number of trips completed by drivers.

This notebook is structured as follows. First, I load and clean the data. Second, I estimate individual labor supply, i.e. how drivers adjust the amount of provided trips given changes in received revenue per trip, obtaining causal estimates through a leave-one-driver-out instrument. Third, I explain in detail how to use this estimate in the tiered incentive scheme. Fourth, for a given reward structure, I use a back-of-envelope calculation to simulate how many additional trips would be completed, along with the expected cost. Finally, I apply a simple optimization over potential reward levels to find the best design given a pre-defined budget.

The goal is not to claim that the result is the final optimal policy, but to show a practical data science application to solve a real-world objective.

## Table of Contents

1. Setting and Data
2. Estimating Drivers' Supply Elasticity
3. Tiered Incentive Scheme
4. Simulating Incentive Effects
5. Optimization


## 1) Setting and Data

### 1.1 Objective

Imagine you are tasked with increasing the number of trips offered by taxi drivers in San Francisco. For example, you might want to increase your market share compared to competitors, or provide a more comprehensive coverage of taxis for citizens and visitors.

You have access to data on all completed rides in the past year, including trip start and end date/time/location, driver IDs, and fare amounts [CITE HERE].


### 1.2 Data

We'll start by loading the data and doing some basic cleaning. In particular, I'm removing trips with missing values and extreme outliers for trip duration, distance, fare, and revenue. I also create driver-day and driver-week panels that we will use in the subsequent analysis. The driver-day panel will be used for the elasticity estimation, while the driver-week panel will be used to calibrate the incentive scheme.


In [1]:
import os
import json
#import warnings
from pathlib import Path

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS

#warnings.filterwarnings('ignore')

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'data' / 'raw').exists() else cwd.parent
RAW = PROJECT_ROOT / 'data' / 'raw'
OUTPUTS = PROJECT_ROOT / 'outputs'
FIGURES = PROJECT_ROOT / 'figures'


pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

#print(f'Project root: {PROJECT_ROOT}')

In [2]:
# Define columns to be loaded
cols = [
    'driver_id',
    'start_time_local', 'end_time_local',
    'pickup_location_latitude', 'pickup_location_longitude',
    'fare_time_milliseconds', 'trip_distance_meters',
    'total_fare_amount'
]


# Load each month and concatenate into a pandas DataFrame
files_raw = [f for f in os.listdir(RAW) if re.match('sf_taxi_trips_.+\\.csv', f)]
parts = []
for fn in files_raw:
    df = pd.read_csv(RAW / fn, usecols=lambda c: c in cols, low_memory=False)
    df['source_file'] = fn
    parts.append(df)
src = pd.concat(parts, ignore_index=True);
del cols, files_raw, parts, fn


# Fix data types and create features
src['start']            = pd.to_datetime(src['start_time_local'], errors='coerce')
src['end']              = pd.to_datetime(src['end_time_local'], errors='coerce')
src['date']             = src['start'].dt.normalize()
src['pickup_hour']      = src['start'].dt.hour

src['trip_hours']       = (src['end'] - src['start']).dt.total_seconds() / 3600
fallback_hours          = pd.to_numeric(src['fare_time_milliseconds'], errors='coerce') / 3_600_000
src['trip_hours']       = src['trip_hours'].where(src['trip_hours'].between(1 / 60, 3), fallback_hours)

src['distance_km']      = pd.to_numeric(src['trip_distance_meters'], errors='coerce') / 1000
src['lat']              = pd.to_numeric(src['pickup_location_latitude'], errors='coerce')
src['lon']              = pd.to_numeric(src['pickup_location_longitude'], errors='coerce')
src['speed_kmh']        = src['distance_km'] / src['trip_hours']

src['fare']             = pd.to_numeric(src['total_fare_amount'], errors='coerce')
src['revenue_per_hour'] = src['fare'] / src['trip_hours']
src['revenue_per_km']   = src['fare'] / src['distance_km']

# Basic cleaning: source trips
basic = src[
    src['driver_id'].notna()
    & src['start'].notna()
    & src['lat'].between(37.20, 38.10)
    & src['lon'].between(-123.0, -121.6)
    & src['fare'].between(2, 300)
    & src['trip_hours'].between(1 / 60, 3)
    & src['distance_km'].between(0, 120)
].copy()
basic['driver_trip_count'] = basic.groupby('driver_id').transform('size')


In [3]:
# ── Driver-day panel (for IV estimation) ─────────────────────────────────
driver_day = (
    basic.groupby(['driver_id', 'date'])
    .agg(
        daily_trips   =('fare',      'count'),
        daily_hours   =('trip_hours','sum'),
        daily_earnings=('fare',      'sum'),
    )
    .reset_index()
)
driver_day['revenue_per_trip']  = driver_day['daily_earnings'] / driver_day['daily_trips']
driver_day['weekday'] = driver_day['date'].dt.dayofweek
driver_day['ym']      = driver_day['date'].dt.to_period('M').astype(str)
driver_day = driver_day[driver_day['daily_trips'] >= 3].reset_index(drop=True)

driver_day['log_trips'] = np.log(driver_day['daily_trips'])
driver_day['log_rpt']   = np.log(driver_day['revenue_per_trip'])


# Week fields for driver-week aggregation
basic['year_week']  = basic['start'].dt.strftime('%G-W%V')
basic['week_start'] = pd.to_datetime(basic['year_week'] + '-1', format='%G-W%V-%u')


# ── Driver-week panel (for tier calibration and simulation) ───────────────
dw_raw = (
    basic.groupby(['driver_id', 'year_week'])
    .agg(
        weekly_trips   =('fare',      'count'),
        weekly_hours   =('trip_hours','sum'),
        weekly_earnings=('fare',      'sum'),
        active_days    =('date',      'nunique'),
    )
    .reset_index()
)
week_to_ym = (
    basic.groupby('year_week')['week_start'].first()
    .dt.to_period('M').astype(str)
    .rename('ym')
)
dw_raw = dw_raw.join(week_to_ym, on='year_week'); del week_to_ym
dw_raw['revenue_per_trip']  = dw_raw['weekly_earnings'] / dw_raw['weekly_trips']
dw_raw['earnings_per_hour'] = dw_raw['weekly_earnings'] / dw_raw['weekly_hours']
driver_week = dw_raw[dw_raw['weekly_trips'] >= 3].reset_index(drop=True)

In [4]:

# Print descriptive statistics
print(f"Number of raw trips:        {len(src):,}")
print(f"Number of clean trips:      {len(basic):,}")
print(f"Period:                     {min(basic['date']):%Y-%m-%d} - {max(basic['date']):%Y-%m-%d}")
print(f"Drivers:                    {basic['driver_id'].nunique():,}")
print(f"Days:                       {(max(basic['date']) - min(basic['date'])).days + 1}")
print(f"Driver-day observations:    {len(driver_day):,}")
print(f"Driver-week observations:   {len(driver_week):,}")

Number of raw trips:        4,129,068
Number of clean trips:      3,846,085
Period:                     2022-12-01 - 2024-05-31
Drivers:                    1,383
Days:                       548
Driver-day observations:    331,629
Driver-week observations:   67,443


### 1.3 Summary statistics

The average driver completes around 55 trips per week over 5.4 active days, for a total driving duration of 15.5 hours. Average revenue per trip is around 34.4 dollars, and average revenue per hour is around 120 dollars. While we don't know how trip revenue compares to drivers' compensation, for the purpose of this exercise I assume that revenue per trip is a proxy for, i.e., proportional to, drivers' wage per trip.

In [5]:
summary_cols = [
    'weekly_trips', 'weekly_hours', 'weekly_earnings',
    'active_days',
     'revenue_per_trip', 'earnings_per_hour'
]
print(driver_week[summary_cols].describe().round(2).to_string())

       weekly_trips  weekly_hours  weekly_earnings  active_days  revenue_per_trip  earnings_per_hour
count      67443.00      67443.00         67443.00     67443.00          67443.00           67443.00
mean          57.01         16.07          1664.35         5.37             34.56             107.73
std           94.83         26.42          2480.28         1.75             17.77              31.48
min            3.00          0.05            11.60         1.00              3.50              13.37
25%           29.00          9.04           920.34         4.00             18.34              80.45
50%           47.00         14.21          1500.70         6.00             32.38             106.09
75%           73.00         20.56          2188.65         7.00             47.30             129.27
max        16027.00       4727.71        430406.42         7.00            228.29             562.94


### 1.4 Descriptive figures

I plot the key variables for the analysis: weekly hours, weekly trips, and revenue per trip. The distributions are right-skewed, with a long tail of high-activity drivers (clipped at the 99th percentile). I highlight the 60th and 80th percentiles. Overall, there is a wide variation in driver activity, which motivates the use of different incentive tiers to target different segments of the driver population.

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

vars_cfg = [
    ('weekly_hours',  'Total active hours',  'Hours per driver-week'),
    ('weekly_trips',  'Completed trips',      'Trips per driver-week'),
    ('revenue_per_trip', 'Revenue per trip',  '$ per trip'),
]

for ax, (col, title, xlabel) in zip(axes, vars_cfg):
    p60 = driver_week[col].quantile(0.60)
    p80 = driver_week[col].quantile(0.80)
    plot_cap = driver_week[col].quantile(0.99)
    plot_values = driver_week[col].clip(upper=plot_cap)
    sns.histplot(plot_values, bins=35, color='steelblue', edgecolor='white', linewidth=0.3, ax=ax)
    ax.axvline(p60, color='dimgray', lw=1.6, linestyle='--', label=f'P60 = {p60:.1f}')
    ax.axvline(p80, color='black', lw=1.6, linestyle=':', label=f'P80 = {p80:.1f}')
    ax.set(title=title, xlabel=f'{xlabel} (top 1% clipped)', ylabel='Driver-weeks')
    ax.legend(fontsize=8)

fig.suptitle('Figure 1: Driver-week distributions', y=1.03)
fig.tight_layout()
fig.savefig(FIGURES / '01_driver_distributions.svg', bbox_inches='tight')
plt.close(fig)
print('Figure saved: figures/01_driver_distributions.svg')

Figure saved: figures/01_driver_distributions.svg


![Figure 1: Driver-week distributions](../figures/01_driver_distributions.svg)


I also plot the number of active days per driver-week. Around half of drivers are active for all 7 days, while only a few work fewer than 5 days. Therefore, baseline activity is already quite high, so the main margin for increasing trips is likely on the intensive rather than extensive margin, i.e., increasing the number of trips per active day rather than increasing the number of active days.

In [7]:
fig, ax = plt.subplots(figsize=(7, 4))
active_day_order = np.arange(1, int(driver_week['active_days'].max()) + 1)
sns.countplot(data=driver_week, x='active_days', order=active_day_order,
              color='steelblue', linewidth=0, ax=ax)
ax.set(
    title='Figure 2: Active days per driver-week',
    xlabel='Number of active days in week',
    ylabel='Number of driver-weeks',
)
fig.tight_layout()
fig.savefig(FIGURES / '02_active_days_distribution.svg', bbox_inches='tight')
plt.close(fig)
print('Figure saved: figures/02_active_days_distribution.svg')

Figure saved: figures/02_active_days_distribution.svg


![Figure 2: Active days per driver-week](../figures/02_active_days_distribution.svg)


## 2) Estimating Drivers' Supply Elasticity

To design effective incentives, we need to understand how drivers adjust their labor supply with respect to their earning opportunities ("wage"). This requires decisions on measurement, empirical strategy, and identification.

### 2.1 Measurement

I measure labor supply using the number of completed trips, and I measure wage using revenue per trip.

Using completed trips has three advantages. First, it functions as a revealed measure of effort, since more availability and more effort selecting good zones for trips likely lead to more trips, but it does not reward idle time. Second, this measure is directly linked to the platform's ultimate goal: more trips mean more service to customers and more revenue for the platform. Third, it is easy for drivers to understand and for the firm to measure. However, it is important to keep drawbacks in mind: drivers may adjust by targeting shorter trips, so trip distance and duration should be monitored as well.

Using revenue per trip as the wage measure is a simplification. I assume that wage is proportional to revenue per trip, i.e., that drivers are incentivized by trip revenue. This is likely an important component in driver compensation. Moreover, when choosing whether to supply the next trip, the expected earning of that next trip is likely a key input in decision-making. However, the time until the next trip also plays a role, which is a drawback, so earnings per hour may be preferable in some specifications.


### 2.2 Empirical Strategy

I use a log-log regression of log number of trips on log revenue per trip at the driver-day level.

Aggregating to a driver-day panel captures the intensive margin of labor supply: the number of trips per day for drivers active on that day. I restrict the sample to days on which drivers are active, i.e., complete at least 3 trips. For simplicity, I do not model the extensive margin here, but it would be a worthwhile expansion in practice.

For estimation, I assume a constant supply elasticity: for a given percentage increase in "wage", the percentage increase in "supply" will be the same regardless of the baseline level. This is a common assumption in labor economics and provides a simple way to summarize the responsiveness of supply to changes in earnings. Empirically, this relationship is captured well by a log-log regression, where the coefficient on log revenue per trip directly gives us the approximate elasticity of supply.

The baseline estimating equation is as follows.

$$
\log Trips_{id} = \varepsilon \cdot \log RPT_{id} + Driver_i + Weekday_{d} + Month_{m(d)} + u_{id}
$$

where $Trips_{id}$ is the number of completed trips by driver $i$ on date $d$, and $RPT_{id}$ is the driver's revenue per trip on that day. Driver fixed effects absorb persistent differences in driver intensity, weekday fixed effects absorb regular weekly patterns, and year-month fixed effects absorb lower-frequency demand and seasonality differences. Since we exploit day-level variation, we cluster standard errors by date to account for potential correlation in unobserved demand shocks across drivers on the same day.


### 2.3 Identification via Hausman-Style Instrumental Variable

The OLS regression above is likely biased because omitted factors may drive both revenue per trip and number of trips. For example, on days with high demand, drivers may complete more trips but face lower revenue per trip due to higher competition and less favorable trip mix, leading to a negative correlation in OLS.

Therefore, I use a same-day leave-one-out instrument that uses variation in city-wide earning conditions to isolate exogenous variation in drivers' own earning opportunities. In particular, I instrument drivers' own revenue per trip with the average revenue per trip of all _other_ drivers on the same day. Intuitively, if conditions are good for other drivers, they are likely good for the given driver as well. However, the leave-one-out construction ensures that the instrument is not mechanically correlated with the driver's own revenue per trip, helping to satisfy the exclusion restriction.

Mathematically, for each driver-day, I construct the instrument as follows:

$$
LOO\_RPT_{id} = \frac{\sum_{j \neq i} RPT_{jd}}{N_d - 1}
$$

The two-stage least squares design is:

$$
\begin{aligned}
\log Trips_{id} &= \varepsilon \cdot \log \widehat{RPT}_{id} + Driver_i + Weekday_d + Month_{m(d)} + u_{id} \\
\log RPT_{id} &= \pi \cdot \log LOO\_RPT_{id} + Driver_i + Weekday_d + Month_{m(d)} + v_{id}
\end{aligned}
$$

The coefficient $\varepsilon$ is the local average treatment effect of revenue per trip, i.e., the elasticity of supply with respect to revenue per trip, for the marginal driver whose revenue per trip is affected by city-wide earning conditions.

The identification assumption is that other drivers' same-day earning opportunities affect a given driver's trips _only_ through that driver's own earning opportunity, conditional on fixed effects. While this is likely an improvement over the initial OLS specification, the design may still be subject to some bias. For example, if drivers observe demand shocks directly (e.g., through shared communication or by visual cues) and adjust their labor supply accordingly, day-level shocks may drive both the instrument and the outcome. In practice, one would therefore want to complement this design by adding flexible controls, exploiting natural experiments such as policy changes, or introducing experimental variation in trip revenues directly.

### 2.4 Estimation

I estimate the 2SLS model using the linearmodels package, adding OLS estimates for comparison. For computational reasons, I residualize before estimation, i.e., demean by fixed effects, which speeds up computations while providing the same coefficient estimates and standard errors. I report coefficient estimates along with standard errors, adjusted R-squared for OLS, and the built-in linearmodels partial first-stage diagnostic for IV specifications.

In [8]:
# Calculate LOO-IV
day_sum = driver_day.groupby('date')['log_rpt'].transform('sum')
day_cnt = driver_day.groupby('date')['log_rpt'].transform('count')
driver_day['log_loo_eq_rpt'] = (day_sum - driver_day['log_rpt']) / (day_cnt - 1)

dd = (
    driver_day[day_cnt >= 2]
    .dropna(subset=['log_trips', 'log_rpt', 'log_loo_eq_rpt'])
    .pipe(lambda df: df[np.isfinite(df['log_loo_eq_rpt'])])
    .copy()
    .reset_index(drop=True)
)


# Within-transform: absorb driver FEs via demeaning (avoids ~1,338-column design matrix)
for col in ['log_trips', 'log_rpt', 'log_loo_eq_rpt']:
    dd[f'{col}_w'] = dd[col] - dd.groupby('driver_id')[col].transform('mean')

clust = dd['date']

# ── helpers ───────────────────────────────────────────────────────────────────
def stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

def fmt(coef, se, pval):
    return f'{coef:.3f}{stars(pval)}', f'({se:.3f})'

def run_iv(fe_terms, data, clusters):
    fe = (f' + {fe_terms}') if fe_terms else ''
    iv = IV2SLS.from_formula(
        f'log_trips_w ~ 1{fe} + [log_rpt_w ~ log_loo_eq_rpt_w]', data=data
    ).fit(cov_type='clustered', clusters=clusters)
    first_stage = iv.first_stage.diagnostics.loc['log_rpt_w']
    return iv, first_stage['f.stat']

# ── OLS ───────────────────────────────────────────────────────────────────────
ols1 = smf.ols('log_trips_w ~ log_rpt_w',                      data=dd).fit(cov_type='cluster', cov_kwds={'groups': clust})
ols2 = smf.ols('log_trips_w ~ log_rpt_w + C(weekday)',         data=dd).fit(cov_type='cluster', cov_kwds={'groups': clust})
ols3 = smf.ols('log_trips_w ~ log_rpt_w + C(weekday) + C(ym)', data=dd).fit(cov_type='cluster', cov_kwds={'groups': clust})

# ── IV ────────────────────────────────────────────────────────────────────────
iv1, iv1_fs = run_iv('',                   dd, clust)
iv2, iv2_fs = run_iv('C(weekday)',         dd, clust)
iv3, iv3_fs = run_iv('C(weekday) + C(ym)', dd, clust)

ELASTICITY = float(iv3.params['log_rpt_w'])

# ── FE-residualized arrays for binscatter (Frisch-Waugh) ─────────────────────
_fe = 'C(weekday) + C(ym)'
_y  = smf.ols(f'log_trips_w      ~ {_fe}', data=dd).fit().resid.values
_x  = smf.ols(f'log_rpt_w        ~ {_fe}', data=dd).fit().resid.values
_z  = smf.ols(f'log_loo_eq_rpt_w ~ {_fe}', data=dd).fit().resid.values
_b1 = float(np.dot(_z, _x) / np.dot(_z, _z))
full_spec = {
    'y': _y, 'x': _x, 'z': _z, 'x_hat': _z * _b1,
    'ols_coef':      float(ols3.params['log_rpt_w']),
    'iv_coef':       ELASTICITY,
    'first_stage_stat': iv3_fs,
    'nobs':          int(iv3.nobs),
}



### 2.5 Results

The preferred specification, IV (6), suggests a supply elasticity of around 1.1, meaning that a one-percent increase in revenue per trip leads to an approximately 1.1 percent increase in the number of completed trips. The estimate is significant at the one-percent level, and the first-stage diagnostic statistic suggests a strong first stage, meaning that the instrument captures meaningful variation in earnings opportunities (although a robust F-statistic like Kleibergen-Paap would be the ideal diagnostic). The estimate is consistent with economic theory, which predicts that higher wages increase labor supply, although the magnitude of the elasticity is on the higher side. By contrast, the OLS estimates confirm the necessity of using a causal design, as the pure correlation between trips and revenue per trip is negative, suggesting that omitted variables produce a spurious correlation, even after controlling for high-dimensional fixed effects. For example, on days with high demand, drivers may complete more trips but face lower revenue per trip due to higher competition and less favorable trip mix, leading to a negative correlation in OLS.

In [9]:
# ── results table ─────────────────────────────────────────────────────────────
Y, N = 'Yes', ''

def row_ols(m, fe_flags):
    c, s = fmt(m.params['log_rpt_w'], m.bse['log_rpt_w'], m.pvalues['log_rpt_w'])
    return [c, s, '', f'{int(m.nobs):,}', f'{m.rsquared_adj:.3f}'] + fe_flags

def row_iv(m, first_stage_stat, fe_flags):
    c, s = fmt(float(m.params['log_rpt_w']), float(m.std_errors['log_rpt_w']), float(m.pvalues['log_rpt_w']))
    return [c, s, f'{first_stage_stat:.0f}', f'{int(m.nobs):,}', ''] + fe_flags

table = pd.DataFrame({
    '':          ['Elasticity', '(SE)', 'First-stage stat.', 'N', 'Adj. R2', 'Driver FE', 'Weekday FE', 'Year-month FE'],
    'OLS (1)': row_ols(ols1, [Y, N, N]),
    'OLS (2)': row_ols(ols2, [Y, Y, N]),
    'OLS (3)': row_ols(ols3, [Y, Y, Y]),
    'IV (4)':  row_iv(iv1, iv1_fs, [Y, N, N]),
    'IV (5)':  row_iv(iv2, iv2_fs, [Y, Y, N]),
    'IV (6)':  row_iv(iv3, iv3_fs, [Y, Y, Y]),
})

print('Table 1: Driver-Day Supply Elasticity Estimates')
print()
print('Dependent Variable: Log(Number of Trips)')
print(table.to_string(index=False))
print()
print('Notes: SE clustered by date. Significance: * p<0.10,  ** p<0.05,  *** p<0.01')
print('First-stage stat. is linearmodels first_stage.diagnostics f.stat.')

# ── save ──────────────────────────────────────────────────────────────────────
model_summary = pd.DataFrame([
    {'spec': n, 'estimator': est, 'Log(Revenue per Trip)': e, 'se': s, 'p_value': p, 'first_stage_stat': fs, 'adj_r2': r2, 'nobs': nobs}
    for n, est, e, s, p, fs, r2, nobs in [
        ('OLS (1)', 'OLS', ols1.params['log_rpt_w'], ols1.bse['log_rpt_w'],       ols1.pvalues['log_rpt_w'], np.nan, ols1.rsquared_adj, int(ols1.nobs)),
        ('OLS (2)', 'OLS', ols2.params['log_rpt_w'], ols2.bse['log_rpt_w'],       ols2.pvalues['log_rpt_w'], np.nan, ols2.rsquared_adj, int(ols2.nobs)),
        ('OLS (3)', 'OLS', ols3.params['log_rpt_w'], ols3.bse['log_rpt_w'],       ols3.pvalues['log_rpt_w'], np.nan, ols3.rsquared_adj, int(ols3.nobs)),
        ('IV (4)',  'IV',  iv1.params['log_rpt_w'],  iv1.std_errors['log_rpt_w'], iv1.pvalues['log_rpt_w'], iv1_fs, np.nan, int(iv1.nobs)),
        ('IV (5)',  'IV',  iv2.params['log_rpt_w'],  iv2.std_errors['log_rpt_w'], iv2.pvalues['log_rpt_w'], iv2_fs, np.nan, int(iv2.nobs)),
        ('IV (6)',  'IV',  iv3.params['log_rpt_w'],  iv3.std_errors['log_rpt_w'], iv3.pvalues['log_rpt_w'], iv3_fs, np.nan, int(iv3.nobs)),
    ]
])
model_summary.to_csv(OUTPUTS / 'driver_day_model_summary.csv', index=False)

Table 1: Driver-Day Supply Elasticity Estimates

Dependent Variable: Log(Number of Trips)
                    OLS (1)   OLS (2)   OLS (3)  IV (4)  IV (5)   IV (6)
       Elasticity -0.461*** -0.462*** -0.449***  -0.092  -0.065 1.059***
             (SE)   (0.007)   (0.007)   (0.005) (0.149) (0.151)  (0.143)
First-stage stat.                                  3455    3638     2429
                N   331,629   331,629   331,629 331,629 331,629  331,629
          Adj. R2     0.102     0.106     0.147                         
        Driver FE       Yes       Yes       Yes     Yes     Yes      Yes
       Weekday FE                 Yes       Yes             Yes      Yes
    Year-month FE                           Yes                      Yes

Notes: SE clustered by date. Significance: * p<0.10,  ** p<0.05,  *** p<0.01
First-stage stat. is linearmodels first_stage.diagnostics f.stat.


Finally, I confirm that the linear functional-form assumption in the log-log specification is appropriate. The following binned scatterplots show the first and second stages separately, along with a fitted OLS line. The first-stage plot shows a strong positive relationship between the instrument (leave-one-out revenue per trip) and the driver's own revenue per trip, confirming relevance. The second-stage plot shows a positive relationship between the instrumented revenue per trip and completed trips, consistent with the estimated elasticity.

In [10]:
def binscatter(x_values, y_values, ax, q=40, title='', xlabel='', ylabel='', color='steelblue'):
    plot_df = pd.DataFrame({'x': x_values, 'y': y_values}).replace([np.inf, -np.inf], np.nan).dropna()
    plot_df['bin'] = pd.qcut(plot_df['x'], q=q, duplicates='drop')
    binned = (
        plot_df.groupby('bin', observed=True)
        .agg(x=('x', 'mean'), y=('y', 'mean'))
        .reset_index(drop=True)
    )
    slope = float(np.polyfit(binned['x'], binned['y'], 1)[0])
    sns.scatterplot(data=binned, x='x', y='y', color=color, s=22, edgecolor=None, alpha=0.9, ax=ax)
    sns.regplot(
        data=binned, x='x', y='y', scatter=False, ci=None, color='crimson', ax=ax,
        line_kws={'lw': 1.6, 'ls': '--', 'label': f'Slope = {slope:.3f}'},
    )
    ax.axhline(0, color='#bbbbbb', lw=0.7, ls=':')
    ax.axvline(0, color='#bbbbbb', lw=0.7, ls=':')
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    ax.legend(fontsize=8)
    return slope

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
binscatter(
    full_spec['z'], full_spec['x'], ax1,
    title='First stage',
    xlabel='Log leave-one-out revenue/trip (demeaned)',
    ylabel='Log own revenue/trip (demeaned)',
)
binscatter(
    full_spec['x_hat'], full_spec['y'], ax2,
    title='Second stage',
    xlabel='Instrumented log revenue/trip (demeaned)',
    ylabel='Log completed trips (demeaned)',
)
fig.suptitle(
    f'Figure 3: Preferred driver-day IV stages | OLS={full_spec["ols_coef"]:.3f} | '
    f'IV={full_spec["iv_coef"]:.3f} | First-stage={full_spec["first_stage_stat"]:.0f} | N={full_spec["nobs"]:,}',
    y=1.03,
    fontsize=10,
)
fig.tight_layout()
fig.savefig(FIGURES / '03_binscatter_functional_form.svg', bbox_inches='tight')
plt.close(fig)
print('Figure saved: figures/03_binscatter_functional_form.svg')


Figure saved: figures/03_binscatter_functional_form.svg


/var/folders/_t/ccptz9gx559517dwn65vyzv80000gn/T/ipykernel_70795/2069899372.py:18: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=8)
/var/folders/_t/ccptz9gx559517dwn65vyzv80000gn/T/ipykernel_70795/2069899372.py:18: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=8)


![Figure 4: Preferred driver-day IV stages](../figures/04_driver_day_iv_stages.svg)


## 3) Tiered Incentive Scheme

### 3.1 Tiered Design

We'll consider a simple tiered scheme that offers cash bonuses for reaching certain trip thresholds in a calendar week. This tiered reward structure is useful because it gives many drivers an attainable next milestone. A driver just below the first threshold can respond to a relatively small reward, while a more active driver may need a larger marginal reward to justify extra work.

The tier design also creates a simple operational lever. Instead of continuously changing prices, an operator can announce clear weekly trip thresholds and payouts, then evaluate how much additional supply appears around those thresholds.

### 3.2 Completed Trips as Incentivized Variable

I use completed trips as the incentivized variable. Completed trips are directly tied to passenger service, are easy to measure in the taxi records, and are harder to manipulate than logged-in time. A driver cannot accumulate trip progress simply by remaining available without serving passengers. The main risk is that taxi drivers select shorter trips to reach thresholds. This is less of a concern because taxi drivers cannot choose trips directly, but customers can choose destinations. It can also be mitigated by adding requirements such as excluding very short trips, setting minimum fare restrictions, and adding guardrails for monitoring, such as trip length as an outcome, cancellations, and service quality.


### 3.3 Example

The following table summarizes the tiered incentive design for an exemplary calibration. I set the example thresholds at the 60th and 80th percentiles of completed weekly trips, rounded to the nearest value of 5, which are around 55 and 80, respectively. I let the first threshold offer a bonus equivalent to 5% of the average weekly wage per driver, while the second threshold offers an additional bonus of 10%. For simplicity, I assume that each taxi driver retains 50% of the fare revenue. Therefore, the total cash bonuses at the two thresholds are approximately 40 and 75 USD.

In [12]:
# Helper function: round to the closest multiple of 'step'
def round_to_5(value):
    return int(5 * np.round(value / 5))

# Setup
TIER1_EARNINGS_INCREASE = 0.05
TIER2_EARNINGS_INCREASE = 0.10

# Calculate trip thresholds: 60th and 80th quantile of weekly trips
threshold_1 = round_to_5(driver_week['weekly_trips'].quantile(0.60))
threshold_2 = round_to_5(driver_week['weekly_trips'].quantile(0.80))

# Calculate threshold rewards: (Threshold) x (Avg. revenue per trip) x (Share retained by driver) x (relative earnings increase)
typical_revenue_per_trip = driver_week['weekly_earnings'].sum() / driver_week['weekly_trips'].sum()
tier1_reward = round_to_5(threshold_1 * typical_revenue_per_trip * 0.5 * TIER1_EARNINGS_INCREASE)
tier2_reward = round_to_5((threshold_2 * typical_revenue_per_trip * 0.5 * TIER2_EARNINGS_INCREASE) - tier1_reward)

tiers = [
    {
        'name': 'Tier 1',
        'threshold': threshold_1,
        'marginal_reward': tier1_reward,
        'total_reward': tier1_reward,
    },
    {
        'name': 'Tier 2',
        'threshold': threshold_2,
        'marginal_reward': tier2_reward,
        'total_reward': tier1_reward +  tier2_reward,
    },
]

for t in tiers:
    t['earnings_increase_at_threshold'] = t['total_reward'] / (t['threshold'] * typical_revenue_per_trip * 0.5)

tier_table = pd.DataFrame(tiers)[[
    'name', 'threshold', 'marginal_reward', 'total_reward', 'earnings_increase_at_threshold'
]].rename(columns={
    'name': 'Tier',
    'threshold': 'Completed trips per week',
    'marginal_reward': 'Marginal bonus ($)',
    'total_reward': 'Total bonus ($)',
    'earnings_increase_at_threshold': 'Total earnings increase at tier',
})

print(tier_table.to_string(index=False, formatters={'Total earnings increase at tier': '{:.1%}'.format}))

  Tier  Completed trips per week  Marginal bonus ($)  Total bonus ($) Total earnings increase at tier
Tier 1                        55                  40               40                            5.0%
Tier 2                        80                  75              115                            9.8%


### 3.4 Simulating Incentive Effects

Next, we can use the estimated supply elasticity from the previous section to simulate how many additional trips the tiered incentive scheme would generate. I assume drivers compare effective revenue per trip with and without the bonus. Reaching threshold $T$ that provides a marginal reward $R$ is equivalent to an effective percentage increase in revenue per trip of $R/(T \cdot r)$ over baseline revenue per trip $r$. Applying the estimated elasticity $\hat\varepsilon$, a driver currently completing $q_{pre}<T$ trips will increase supply to $T$ trips if:

$$
\begin{aligned}
&q_{pre} + q_{pre} \cdot \left(\hat\varepsilon \cdot \frac{R}{T \cdot r}\right) \geq T \\
\Leftrightarrow \qquad &q_{pre} \geq \frac{T}{1 + \hat\varepsilon \cdot \frac{R}{T \cdot r}}
\end{aligned}
$$

Thus, each tier defines a lower bound on current weekly trips above which a driver is predicted to reach the next milestone. Drivers below the response window are left unchanged in this simple simulation. Applying this rule to our data, we estimate that driver-week observations with 53 and 54 trips would increase their supply to 55 trips, while those between 75 and 79 trips would increase to 80 trips. Overall, 3,523 driver-week observations will increase their supply to reach the next threshold.

In [14]:
for t in tiers:
    t['lower_bound'] = t['threshold'] / (1 + ELASTICITY * t['earnings_increase_at_threshold'])

print(f"Elasticity used for simulation: {ELASTICITY:.3f}")
print(f"Baseline wage per trip used for calibration: ${typical_revenue_per_trip * 0.5:.2f}")
print()
print(f"{'Tier':<8} {'Response window':<34}  {'Driver-weeks':>14}")
print('-' * 81)
for t in tiers:
    n = ((driver_week['weekly_trips'] >= t['lower_bound']) & (driver_week['weekly_trips'] < t['threshold'])).sum()
    print(
        f"{t['name']:<8} "
        f"[{t['lower_bound']:6.1f}, {t['threshold']:6.0f}) trips -> {t['threshold']:.0f} {n:>14,}"
    )

Elasticity used for simulation: 1.059
Baseline wage per trip used for calibration: $14.60

Tier     Response window                       Driver-weeks
---------------------------------------------------------------------------------
Tier 1   [  52.2,     55) trips -> 55          1,461
Tier 2   [  72.4,     80) trips -> 80          2,972


To visualize the adjustment, we can plot the distribution of weekly trips before and after the incentive scheme. We can see that the distribution becomes more concentrated around the thresholds, with a noticeable increase in the number of driver-weeks at 55 and 80 trips.


To estimate the business impact, we can calculate the number of additional trips, number of additional fare revenue, and the total cost of the bonuses. We assume that the additional trips generate the same average revenue per trip as the baseline, and that from the half of the additional fare revenue that is _not_ retained by the driver, the platform retains 100% as profit (since these are incremental trips with no meaningful overhead).

Under this simulation, the incentive scheme generates around 8,300 additional trips per week, which corresponds to around $270 per additional trip.

In [15]:
sim = driver_week.copy()
sim['trips_post'] = sim['weekly_trips'].astype(float)

# Apply the higher tier first so drivers close to Tier 2 are moved to the higher milestone.
for t in sorted(tiers, key=lambda d: d['threshold'], reverse=True):
    mask = (sim['weekly_trips'] >= t['lower_bound']) & (sim['weekly_trips'] < t['threshold'])
    sim.loc[mask, 'trips_post'] = t['threshold']

# Calculate accrued bonuses
sim['bonus'] = 0.0
for t in tiers:
    sim.loc[sim['trips_post'] >= t['threshold'], 'bonus'] += t['marginal_reward']

# Calculate total bonus, additional trips, additional company earnings, total co
total_bonus = sim['bonus'].sum()

# Calculate number of additional trips
base_trips       = sim['weekly_trips'].sum()
additional_trips = sim['trips_post'].sum() - base_trips

# Calculate additional company earnings:
# (Additional trips) x (Avg. revenue per trip) x (Share not retained by driver) x (100%)
base_revenue       = base_trips * typical_revenue_per_trip
additional_revenue = additional_trips * typical_revenue_per_trip * 0.5

# Calculate total cost for company:
# (Total Bonus) - (Additional Revenue)
total_cost = total_bonus - additional_revenue

# Calculate efficiency: Total cost per additional trip
total_cost_per_additional_trip = total_cost / additional_trips

# Print results
print(f'Additional trips:            {additional_trips:,.0f}     (+{additional_trips / base_trips:.2%})')
print(f'Additional revenue:          {additional_revenue:,.1f} (+{additional_revenue / base_revenue:.2%})')
print(f'Total bonus paid:           ${total_bonus:,.0f}')
print(f'Total program cost:         ${total_cost:,.0f}')
print(f'Cost per additional trip:   ${total_cost_per_additional_trip:,.2f}')

Additional trips:            14,233     (+0.37%)
Additional revenue:          207,760.2 (+0.19%)
Total bonus paid:           $2,446,360
Total program cost:         $2,238,600
Cost per additional trip:   $157.28


In [17]:
plot_cap = sim[['weekly_trips', 'trips_post']].to_numpy().ravel()
plot_cap = np.nanquantile(plot_cap, 0.99)
bins = np.linspace(0, plot_cap, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, col, title, color in [
    (ax1, 'weekly_trips', 'Baseline', 'steelblue'),
    (ax2, 'trips_post', 'With trip-tier bonus', 'tomato'),
]:
    plot_values = sim[col].clip(upper=plot_cap)
    ax.hist(plot_values, bins=bins, color=color, edgecolor='white', linewidth=0.25)
    for t, c in zip(tiers, ['#555555', '#111111']):
        ax.axvline(t['threshold'], color=c, lw=1.5, linestyle='--', label=f"{t['name']} ({t['threshold']:.0f} trips)")
    ax.set(xlabel='Completed trips per driver-week (top 1% clipped)', title=title)
ax1.set_ylabel('Driver-weeks')
fig.suptitle('Figure 5: Completed trips without and with trip-tier bonus', y=1.03)
fig.tight_layout()
fig.savefig(FIGURES / '04_trip_tier_simulation.svg', bbox_inches='tight')
plt.close(fig)
print('Figure saved: figures/04_trip_tier_simulation.svg')

Figure saved: figures/04_trip_tier_simulation.svg


![Figure 5: Completed trips without and with trip-tier bonus](../figures/05_trip_tier_simulation.svg)


## 4) Optimization

The current scheme is quite expensive, so let's see whether we can optimize it a bit by adjusting the thresholds and rewards. We'll use a a grid search over thresholds and reward amounts to find the combination that maximizes additional trips while keeping total program cost under $2,000,000. To keep things simple, we will only consider adjustments with two tiers.